# Baselines - Part 3: UNet width selection, B1, and B2 on fold I

This notebook compares the two UNet widths (`32-64-128` and `64-128-256`) on TESSERA 2019 at 100% and 1% label fractions across seeds 0, 1, and 2, selects the width using validation patches only (fold 4), trains the pixel MLP baseline (B1) under the same setup, and summarizes test performance for B0 (linear head), B1 (pixel MLP), and B2 (UNet) across all valid pixels, parcel boundaries, and parcel cores.

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS, RUNS_CSV = DRIVE / "runs", DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "baselines"
B1 = "configs/baselines/b1_mlp_t19.yaml"
B2 = "configs/baselines/b2_unet_t19.yaml"

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 data to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Train B2 with both UNet widths
Train `unet-t19-w32` (`32-64-128`) and `unet-t19-w64` (`64-128-256`) at 100% and 1% label fractions over seeds 0, 1, and 2. Completed runs on Drive are skipped automatically, and missing per-patch validation confusion matrices (`cm_val_per_patch.npz`) are computed from `best.pt`.

In [ ]:
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0 1 2 --label-fracs 1.0 0.01

In [ ]:
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0 1 2 --label-fracs 1.0 0.01 --widths 64 128 256

## 5. Select UNet width on validation patches
`scripts/select_unet_width.py` computes the paired patch-bootstrap difference `mIoU(w64) - mIoU(w32)` on fold 4 validation patches averaged across seeds. We keep `64-128-256` only if its 95% confidence interval is strictly above zero for at least one label fraction and never strictly below zero; otherwise we keep `32-64-128`. The selected width is saved to `results/baselines/unet_width_foldI.json`.

In [ ]:
!python scripts/select_unet_width.py --runs-dir {RUNS} --out {OUT}

from IPython.display import Image, display
decision = json.loads((OUT / "unet_width_foldI.json").read_text())
B2_VARIANT = decision["chosen_variant"]
display(Image(str(OUT / "unet_width_foldI_val_curves.png")))
print("B2 from now on:", B2_VARIANT)

## 6. Train B1 (TESSERA 2019 + pixel MLP)
Train the per-pixel MLP (`1x1` convolutions without spatial context) at 100% and 1% label fractions across seeds 0, 1, and 2 using the same optimizer budget and loss as B2.

In [ ]:
!python scripts/train_head.py --config {B1} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0 1 2 --label-fracs 1.0 0.01

## 7. Test set summary and paired bootstrap comparisons
Aggregate test mIoU over the 3 seeds with patch-bootstrap confidence intervals across all valid pixels, parcel boundaries, and parcel cores, and compute paired differences (`B2 - B1`, `B2 - B0`, `B1 - B0`). Comparing B2 against B1 isolates the contribution of spatial context on identical TESSERA 2019 inputs.

In [ ]:
from scripts.summarize_runs import summarize_runs

B0, B1_TAG, B2_TAG = "lin_t19", "base_mlp-t19", f"base_{B2_VARIANT}"
setups = {1.0: ([B0, B1_TAG, B2_TAG], [f"{B2_TAG}:{B1_TAG}", f"{B2_TAG}:{B0}", f"{B1_TAG}:{B0}"]),
          0.01: ([B1_TAG, B2_TAG], [f"{B2_TAG}:{B1_TAG}"])}
pd.set_option("display.width", 220)
for frac, (variants, pairs) in setups.items():
    name = f"baselines_foldI_{round(frac * 100):03d}"
    per_seed, summary, deltas = summarize_runs(RUNS, OUT, name=name, variants=variants, pairs=pairs,
                                               split="I", label_frac=frac, seeds=(0, 1, 2))
    print(f"===== {frac:.0%} labels =====")
    print(summary[["variant", "seeds", "miou_mean", "miou_std", "ci_low", "ci_high", "boundary_mean",
                   "core_mean", "val_miou_mean", "n_params", "train_seconds_mean"]].round(4).to_string(index=False))
    print("\npaired bootstrap, delta = mIoU(variant) - mIoU(versus), averaged over seeds:")
    print(deltas.round(4).to_string(index=False))
    display(Image(str(OUT / f"{name}_per_class.png")))
    print()